In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F


In [0]:
#try:

# 1. Configuración de rutas o tablas en Unity Catalog
# Bronze: Datos crudos (raw) tal como vienen del origen
# Silver: Datos conformados (limpios y normalizados)
bronze_table = "bronze.brz_inmuebles"
silver_table = "silver.slv_inmuebles"

# 2. Lectura en modo BATCH
# Se usa .read para procesamiento por lotes (.readStream para stream)
df_bronze = spark.table(bronze_table)

# 3. Transformaciones (Limpieza y Enriquecimiento)
df_silver = df_bronze \
    .filter(F.col("Referencia").isNotNull()) \
    .dropDuplicates(["Referencia"]) \
    .withColumn("FechaAlta", F.date_format(F.col("FechaAlta"), "yyyy-MM-dd").cast('Date')) \
    .withColumn("Fecha_Venta", F.date_format(F.col("Fecha_Venta"), "yyyy-MM-dd").cast('Date'))

                                                                               
# 4. Escritura en la capa Silver
# Se suele usar .mode("append") para añadir nuevos lotes o "overwrite" para reemplazo total
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)
#except Exception as e:
#    print(f"Tipo de error: {type(e).__name__}, Mensaje: {e}")

In [0]:
%sql
select * from silver.slv_inmuebles